#### Importing libraries

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


#### Data Loading

In [3]:
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


#### Data Cleaning

In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
# lists of columns by type
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [6]:
# filling missing values
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0)

In [7]:
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

#### most frequent observation (mode) for the column ```industry```

In [8]:
print(df['industry'].value_counts())
print()
print('Answer Q1 (mode):', df['industry'].mode()[0])

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Answer Q1 (mode): technology


The most freq observation mode was technology using ```value_counts()```

#### Biggest Correlation by using Correlation matrix 

In [9]:
#Correlation matrix
corr_matrix = df[numerical].corr()
corr_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [10]:
# comparing the correlation features
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(f'{a} and {b}: {corr_matrix.loc[a, b]:.4f}')

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [11]:
# finding the best correlation 
best_pair = max(pairs, key=lambda p: abs(corr_matrix.loc[p[0], p[1]]))
print('answer Q2:', best_pair)

answer Q2: ('interaction_count', 'lead_score')


the biggest correlation based on correlation matrix is ```interaction_count1``` and ```lead_score```

#### Splitting the data
splitting into 3 parts. train, validation, test

In [12]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print('train:', len(df_train), '| val:', len(df_val), '| test:', len(df_test))


train: 3000 | val: 1000 | test: 1000


#### Calculating Mutual Information score between converted and other categorical variables

In [13]:
scores = {}
for col in ['industry', 'location', 'lead_source', 'employment_status']:
    score = mutual_info_score(df_train[col], y_train)
    scores[col] = round(score, 2)
scores

{'industry': 0.0,
 'location': 0.0,
 'lead_source': 0.03,
 'employment_status': 0.02}

In [14]:
best_col = max(scores, key=scores.get)
print('Answer Q3:', best_col, scores[best_col])

Answer Q3: lead_source 0.03


the variable that has the biggest mutual information score is ```lead_souce```

#### logreg using one-hot encoding

In [15]:
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print('feaature matrix shape:', X_train.shape)

feaature matrix shape: (3000, 28)


In [16]:
# model training
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print('Accuracy:', accuracy)
print('Answer:', round(accuracy, 2))

Accuracy: 0.645
Answer: 0.65


The accuracy produced was 0.65

#### Feature Elimination to find the least useful feature

In [17]:
def train_and_score(features):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_va = dv.transform(df_val[features].to_dict(orient='records'))

    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    return accuracy_score(y_val, m.predict(X_va))

all_features = categorical + numerical
original_accuracy = train_and_score(all_features)
print('Original accuracy:', original_accuracy)

Original accuracy: 0.645


In [18]:
# making lists of all features to check accuracy and their differences
results = []
for feature in all_features:
    features_without = [f for f in all_features if f != feature]
    acc = train_and_score(features_without)
    results.append(
        {'feature_removed': feature,
         'accuracy': acc,
         'accuracy_diff': original_accuracy - acc
        }
    )
results_df = pd.DataFrame(results)
results_df

,feature_removed,accuracy,accuracy_diff
0,lead_source,0.642,0.003
1,industry,0.645,0.000
2,employment_status,0.645,0.000
3,location,0.645,0.000
4,annual_income,0.724,-0.079
5,number_of_courses_viewed,0.643,0.002
6,interaction_count,0.601,0.044
7,lead_score,0.644,0.001


In [21]:
options = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
subset = results_df[results_df['feature_removed'].isin(options)].copy()

subset['abs_difference'] = subset['accuracy_diff'].abs()
print(subset[['feature_removed', 'accuracy_diff', 'abs_difference']])

smallest = subset.loc[subset['abs_difference'].idxmin(), 'feature_removed']
print()
print(smallest)
print(subset.loc[subset['accuracy_diff'].idxmin(), 'feature_removed'])

            feature_removed  accuracy_diff  abs_difference
0               lead_source          0.003           0.003
5  number_of_courses_viewed          0.002           0.002
6         interaction_count          0.044           0.044

number_of_courses_viewed
number_of_courses_viewed


#### finding the param to find the best accuracy

In [22]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_val, model.predict(X_val))
    print(f'C = {C}: accuracy = {round(acc, 3)}')

C = 1e-06: accuracy = 0.598
C = 1e-05: accuracy = 0.598
C = 0.0001: accuracy = 0.613
C = 0.001: accuracy = 0.645
